# Laboratorium 9: Detektor Harrisa

**Politechnika Świętokrzyska**  
**Centrum Laserowych Technologii Metali**  
**Automatyka i Robotyka - 2 semestr II stopnia**

---

## Spis treści
1. [Teoria](#teoria)
2. [Kod startowy](#kod-startowy)
3. [Zadania do wykonania](#zadania)
4. [Przykłady użycia](#przyklady)


## 1. Teoria <a id='teoria'></a>

### 1.1 Wprowadzenie

**Detektor Harrisa** to klasyczny algorytm wykrywania narożników w obrazach (Harris & Stephens, 1988).

**Narożnik** = punkt z znaczącą zmianą intensywności w wielu kierunkach.

**Zastosowania:** śledzenie obiektów, dopasowywanie obrazów, rekonstrukcja 3D, nawigacja robotów.


### 1.2 Matematyka

**Zmiana intensywności przy przesunięciu $(u, v)$:**

$$E(u, v) = \sum_{x,y} w(x,y) \left[ I(x+u, y+v) - I(x,y) \right]^2$$

**Aproksymacja Taylora:**

$$E(u, v) \approx \begin{bmatrix} u & v \end{bmatrix} M \begin{bmatrix} u \\ v \end{bmatrix}$$

**Macierz struktury:**

$$M = \begin{bmatrix} A & C \\ C & B \end{bmatrix} = \begin{bmatrix} G*I_x^2 & G*(I_xI_y) \\ G*(I_xI_y) & G*I_y^2 \end{bmatrix}$$

gdzie $G$ = filtr Gaussa, $I_x, I_y$ = gradienty.


### 1.3 Funkcja odpowiedzi Harrisa

$$R = \det(M) - k \cdot \text{trace}(M)^2 = AB - C^2 - k(A+B)^2$$

**Parametr k (0.04-0.06):**
- **Mniejsze k** → wyższa czułość → więcej narożników
- **Większe k** → niższa czułość → mniej narożników (tylko silne)
- **Domyślnie: k = 0.04**

**Interpretacja:**
- $R > 0$ i duże → **narożnik**
- $R < 0$ → krawędź
- $|R|$ małe → płaski obszar


### 1.4 Algorytm

1. Konwersja do skali szarości
2. Obliczenie gradientów (Sobel): $I_x$, $I_y$
3. Macierz struktury: $A = G*I_x^2$, $B = G*I_y^2$, $C = G*(I_xI_y)$
4. Odpowiedź Harrisa: $R = AB - C^2 - k(A+B)^2$
5. Progowanie: $R > \text{threshold}$
6. Non-Maximum Suppression (NMS)


## 2. Kod startowy <a id='kod-startowy'></a>

### 2.1 Import bibliotek


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from scipy import ndimage
from skimage import io

plt.rcParams['figure.figsize'] = (15, 8)
plt.rcParams['image.cmap'] = 'gray'

### 2.2 Funkcje pomocnicze


In [ ]:
def display_images(images, titles=None, figsize=(15, 5)):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for i, (img, ax) in enumerate(zip(images, axes)):
        ax.imshow(img, cmap='gray')
        ax.axis('off')
        if titles and i < len(titles):
            ax.set_title(titles[i])
    plt.tight_layout()
    plt.show()

def display_corners(image, corners, marker_size=20):
    plt.figure(figsize=(10, 8))
    if len(image.shape) == 3:
        plt.imshow(image)
    else:
        plt.imshow(image, cmap='gray')
    y, x = np.where(corners)
    plt.plot(x, y, 'r+', markersize=marker_size, markeredgewidth=2)
    plt.title(f'Wykryte narożniki: {len(x)}')
    plt.axis('off')
    plt.show()

### 2.3 Operatory Sobela


In [ ]:
# Operatory Sobela
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)
sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=np.float32)

print("Operator Sobel X:")
print(sobel_x)
print("\nOperator Sobel Y:")
print(sobel_y)

### 2.4 Obrazy testowe


In [ ]:
# Obraz testowy 1: Prostokąty
test_img = np.zeros((300, 300), dtype=np.uint8)
cv2.rectangle(test_img, (50, 50), (120, 120), 255, -1)
cv2.rectangle(test_img, (150, 80), (250, 150), 255, -1)
cv2.rectangle(test_img, (80, 180), (200, 250), 255, -1)

# Obraz testowy 2: Lena
lena_url = 'https://raw.githubusercontent.com/kbor89/TWiPO/main/images/lenna.png'
lena = io.imread(lena_url)
if len(lena.shape) == 3:
    lena_gray = cv2.cvtColor(lena, cv2.COLOR_RGB2GRAY)
else:
    lena_gray = lena

display_images([test_img, lena_gray], ['Obraz testowy 1: Prostokąty', 'Obraz testowy 2: Lena'])
print(f"Rozmiar obrazu testowego 1: {test_img.shape}")
print(f"Rozmiar obrazu testowego 2: {lena_gray.shape}")

## 3. Zadania do wykonania <a id='zadania'></a>

### Zadanie 1: Obliczanie gradientów

Zaimplementuj funkcję obliczającą gradienty obrazu.


In [ ]:
def compute_gradients(image):
    """
    Oblicza gradienty obrazu w kierunkach X i Y.
    
    Args:
        image: obraz w skali szarości
    Returns:
        Ix, Iy: gradienty
    """
    # TODO: Konwertuj do float32
    # TODO: Zastosuj splot z sobel_x i sobel_y (użyj cv2.filter2D lub ndimage.convolve)
    # TODO: Zwróć Ix, Iy
    pass

In [ ]:
# Test na obrazie prostokątów
Ix, Iy = compute_gradients(test_img)
if Ix is not None and Iy is not None:
    display_images([test_img, Ix, Iy], ['Oryginał', 'Gradient X', 'Gradient Y'])
    print(f"Zakres Ix: [{Ix.min():.2f}, {Ix.max():.2f}]")
    print(f"Zakres Iy: [{Iy.min():.2f}, {Iy.max():.2f}]")

---

### Zadanie 2: Macierz struktury

Zaimplementuj funkcję obliczającą elementy macierzy struktury.


In [ ]:
def compute_structure_tensor(Ix, Iy, sigma=1.5):
    """
    Oblicza elementy macierzy struktury.
    
    Args:
        Ix, Iy: gradienty
        sigma: parametr wygładzania Gaussa
    Returns:
        A, B, C: elementy macierzy
    """
    # TODO: Oblicz Ix^2, Iy^2, Ix*Iy
    # TODO: Zastosuj wygładzanie Gaussa (cv2.GaussianBlur lub ndimage.gaussian_filter)
    # TODO: Zwróć A, B, C
    pass

In [ ]:
# Test na obrazie prostokątów
Ix, Iy = compute_gradients(test_img)
A, B, C = compute_structure_tensor(Ix, Iy, sigma=1.5)
if A is not None and B is not None and C is not None:
    display_images([A, B, C], ['A = G*Ix²', 'B = G*Iy²', 'C = G*(Ix*Iy)'])
    print(f"Zakres A: [{A.min():.2f}, {A.max():.2f}]")
    print(f"Zakres B: [{B.min():.2f}, {B.max():.2f}]")
    print(f"Zakres C: [{C.min():.2f}, {C.max():.2f}]")

---

### Zadanie 3: Funkcja odpowiedzi Harrisa

Zaimplementuj funkcję odpowiedzi Harrisa.


In [ ]:
def harris_response(A, B, C, k=0.04):
    """
    Oblicza funkcję odpowiedzi Harrisa.
    
    Args:
        A, B, C: elementy macierzy struktury
        k: parametr czułości (0.04-0.06)
    Returns:
        R: mapa odpowiedzi
    """
    # TODO: det = A*B - C^2
    # TODO: trace = A + B
    # TODO: R = det - k * trace^2
    # TODO: Zwróć R
    pass

In [ ]:
# Test na obrazie prostokątów
Ix, Iy = compute_gradients(test_img)
A, B, C = compute_structure_tensor(Ix, Iy, sigma=1.5)
R = harris_response(A, B, C, k=0.04)
if R is not None:
    display_images([test_img, R], ['Oryginał', 'Odpowiedź Harrisa'])
    print(f"Zakres R: [{R.min():.2e}, {R.max():.2e}]")
    print(f"Liczba dodatnich wartości: {np.sum(R > 0)}")

---

### Zadanie 4: Non-Maximum Suppression

Zaimplementuj tłumienie niemaksimów.


In [ ]:
def non_maximum_suppression(response, window_size=5):
    """
    Tłumienie niemaksimów - zachowuje tylko lokalne maksima.
    
    Args:
        response: mapa odpowiedzi Harrisa
        window_size: rozmiar okna
    Returns:
        Mapa z lokalnymi maksimami
    """
    # Znajdź lokalne maksima
    local_max = ndimage.maximum_filter(response, size=window_size)
    
    # Zachowaj tylko piksele równe lokalnemu maksimum
    result = response * (response == local_max)
    
    return result

In [ ]:
# Test tłumienia niemaksimów
Ix, Iy = compute_gradients(test_img)
A, B, C = compute_structure_tensor(Ix, Iy, sigma=1.5)
R = harris_response(A, B, C, k=0.04)
R_nms = non_maximum_suppression(R, window_size=5)
if R_nms is not None:
    display_images([R, R_nms], ['Przed NMS', 'Po NMS'])
    print(f"Niezerowe piksele przed NMS: {np.sum(R > 0)}")
    print(f"Niezerowe piksele po NMS: {np.sum(R_nms > 0)}")

---

### Zadanie 5: Kompletny detektor Harrisa

Połącz wszystkie kroki w jedną funkcję.


In [ ]:
def harris_corner_detector(image, k=0.04, sigma=1.5, threshold=0.01, window_size=5):
    """
    Kompletny detektor narożników Harrisa.
    
    Args:
        image: obraz wejściowy
        k: parametr czułości (0.04-0.06)
        sigma: wygładzanie Gaussa
        threshold: próg detekcji (0.0-1.0)
        window_size: rozmiar okna NMS
    Returns:
        corners: maska binarna z narożnikami
        response: mapa odpowiedzi
    """
    # TODO: 1. Oblicz gradienty
    # TODO: 2. Oblicz macierz struktury
    # TODO: 3. Oblicz odpowiedź Harrisa
    # TODO: 4. Zastosuj NMS
    # TODO: 5. Progowanie: threshold_value = threshold * R_nms.max()
    # TODO: 6. Utwórz maskę binarną
    # TODO: Zwróć corners, response
    pass

In [ ]:
# Test na obrazie prostokątów
corners, response = harris_corner_detector(test_img, k=0.04, threshold=0.01)
if corners is not None:
    display_corners(test_img, corners)
    print(f"Wykryto {np.sum(corners > 0)} narożników")

In [ ]:
# Test na obrazie Leny
corners_lena, response_lena = harris_corner_detector(lena_gray, k=0.04, sigma=1.5, threshold=0.01)
if corners_lena is not None:
    display_corners(lena_gray, corners_lena, marker_size=10)
    print(f"Wykryto {np.sum(corners_lena > 0)} narożników na obrazie Leny")

## 4. Przykłady użycia <a id='przyklady'></a>

### Przykład 1: Wpływ parametru k


In [ ]:
# Porównanie różnych wartości k na obrazie Leny
print("Wpływ parametru k na liczbę wykrytych narożników:\n")
for k_val in [0.04, 0.05, 0.06]:
    corners, _ = harris_corner_detector(lena_gray, k=k_val, threshold=0.01)
    if corners is not None:
        num = np.sum(corners > 0)
        print(f"k = {k_val}: {num} narożników")
        display_corners(lena_gray, corners, marker_size=8)

### Przykład 2: Wpływ parametru sigma


In [ ]:
# Porównanie różnych wartości sigma
print("Wpływ parametru sigma na liczbę wykrytych narożników:\n")
for sigma_val in [0.5, 1.5, 3.0]:
    corners, _ = harris_corner_detector(lena_gray, sigma=sigma_val, threshold=0.01)
    if corners is not None:
        num = np.sum(corners > 0)
        print(f"sigma = {sigma_val}: {num} narożników")
        display_corners(lena_gray, corners, marker_size=8)

### Przykład 3: Porównanie z OpenCV


In [ ]:
# Porównanie własnej implementacji z OpenCV
# Własna implementacja
corners_custom, _ = harris_corner_detector(test_img, k=0.04, threshold=0.01)

# OpenCV
dst = cv2.cornerHarris(test_img.astype(np.float32), blockSize=2, ksize=3, k=0.04)
dst = cv2.dilate(dst, None)
corners_opencv = np.zeros_like(test_img)
corners_opencv[dst > 0.01 * dst.max()] = 255

if corners_custom is not None:
    display_images([corners_custom, corners_opencv], 
                   ['Własna implementacja', 'OpenCV'])
    print(f"Własna implementacja: {np.sum(corners_custom > 0)} narożników")
    print(f"OpenCV: {np.sum(corners_opencv > 0)} narożników")

## Podsumowanie

### Poznane koncepcje:

1. **Detektor Harrisa** - wykrywanie narożników przez analizę gradientów
2. **Macierz struktury** - opisuje lokalne zmiany intensywności
3. **Parametr k** - kontroluje czułość detektora (0.04-0.06)
4. **Non-Maximum Suppression** - eliminacja słabych odpowiedzi

### Parametry do eksperymentowania:

- **k**: 0.04 (wysoka czułość) → 0.06 (niska czułość)
- **sigma**: 0.5 (słabe wygładzanie) → 3.0 (silne wygładzanie)
- **threshold**: 0.001 (wiele narożników) → 0.05 (tylko silne)
- **window_size**: 3 (małe okno NMS) → 7 (duże okno NMS)

### Dalsze zastosowania:

- Śledzenie punktów w wideo
- Dopasowywanie obrazów (image matching)
- SLAM (Simultaneous Localization and Mapping)
- Rozpoznawanie obiektów
